<a href="https://colab.research.google.com/github/kmaranga/Deep-Learning/blob/main/Deep_Implicit_Layers_Intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Most fundamentally, implicit form layers separate the solution procedure of the layer from the definition of the layer itself. This level of modularity has proven extremely useful in a number of domains. Indeed, because we rarely find exact solutions to e.g., algebraic or differential equations, the different solutions methods can be evaluated objectively against each other based upon how well they satisfy the conditions that the layer is attempting to satisfy.

Fortunately, as we will illustrate below, and highlight several times within this tutorial, implicit layers have the notable advantage that we can use the implicit function theorem to directly compute gradients at the solution point of these equations, without having to store any intermediate variables along the way. This vastly improves the memory consumption and often the numerical accuracy of these methods, providing another notable benefit for implicit models in the setting of deep learning in particular.

Implicit layers have been used to:

Solve arbitrary structured convex problems (using the cvxpy library) in a differentiable manner.
Solve smoothed relaxtions of combinatorial optimization problems, such as graph cuts, satisfiability, and many others.
Integrate differential equations as layers in deep networks (with numerous applications in and of itself, such as integrating continuous time observations, or approximating continuous version of traditional residual networks).
Create architectures for efficient representation of smooth densities, for use in generative modeling an beyond.
Achieve performance on par with state-of-the-art Transformer models (at the same parameter count), for language modeling and on par with state-of-the-art computer vision architectures on tasks such as classification and semantic segmentation.

First implicit(network) layer, defined via a fixed point iteration. Essentially is a version of recurrent backpropagation, and also the approach underlying deep equilibrium(DEQ) models.

A fixed point iteration layer, can be interpreted as a simple recurrent network, where z is the hidden layer, and where we repeatedly apply the network to the same input x. It can also reap the benefits of a "deep neural network" while only having the params 'W' of a 'single' layer.

N.B. tanh activation funcs ensure the values of z(the output) never leave the range[-1, +1].
p.s: will also cover issues of "existence" and "uniqueness" later.

As a start, consider the simplest implementation of such a layer, which simply repeats the fixed point iteration to converge, via normal autograd. With normal autograd, each intermediate layer has to be stored in memory, & the backward pass proceeds similarly over the same iterations, but in REVERSE order. Below, I make a composition of tanh and linear layer, and store the most recent iteration count and error(for simplicity)

In [1]:
import torch
import torch.nn as nn

In [2]:
class TanhFixedPointLayer(nn.Module):
  def __init__(self, out_features, tol = 1e-4, max_iter = 50):
    super().__init__()
    self.linear = nn.Linear(out_features, out_features, bias = False) #784 bc the MNIST dataset has the shape 100, 784
    self.tol = tol
    self.max_iter = max_iter

  #forward pass
  def forward(self, x):
    #init output z to be zero
    z = torch.zeros(x.shape[0], self.linear.out_features, device = x.device)
    self.iterations = 0

    #iterate till convergence
    while self.iterations < self.max_iter:
      z_next = torch.tanh(self.linear(z) + x) #huh? removed x from the linear layer?
      self.err = torch.norm(z - z_next) #calculate the error by finding the difference between iteration outputs
      z = z_next #update step
      self.iterations += 1 #update iterations - continues till max_iter
      if self.err < self.tol:
        break #break out of the while loop if our error is below our tolerance


    return z  #return our output

In [3]:
#now run our layer above on a random output:
layer = TanhFixedPointLayer(50)
X = torch.randn(10, 50)
Z = layer(X)
print(f"Terminated after {layer.iterations} iterations with error {layer.err}")

Terminated after 15 iterations with error 5.44105569133535e-05


Now, we'll run the layer with a real model - MNIST! (or rather a model trained on MNIST), and then add a linear input layer before the fixed pt layer.

In [4]:
#import MNIST dataset:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

mnist_train = datasets.MNIST(root = './data', train = True, download = True, transform = transforms.ToTensor())
mnist_test = datasets.MNIST(root = './data', train = False, download = True, transform = transforms.ToTensor())

train_loader = DataLoader(mnist_train, batch_size = 784, shuffle = True)
test_loader = DataLoader(mnist_test, batch_size = 784, shuffle = False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
#we now construct the simple model with the fixed point layer:
import torch.optim as optim

torch.manual_seed(0)
model = nn.Sequential(
    nn.Flatten(),
    TanhFixedPointLayer(784, max_iter = 200),
    nn.Linear(784, 100), #rather than 784, 100
    #nn.ReLU()
     ).to(device)

#optimiser step using stochastic gradient descent and a learning rate of 1e-1
opt = optim.SGD(model.parameters(), lr = 1e-1)

In [14]:
#generic func to run a single training or test epoch:
from tqdm.notebook import tqdm

def epoch(loader, model, opt = None, monitor = None):
  total_loss, total_err, total_monitor = 0., 0., 0.
  model.eval() if opt is None else model.train() #if the model has no object, evaluate it, else keep training
  for X, y in tqdm(loader, leave = False):
    X, y = X.to(device), y.to(device)
    yp = model(X)
    loss = nn.CrossEntropyLoss()(yp, y) #compute cross entropy loss of model(X) and y

    if opt:
      opt.zero_grad() #zero gradients
      loss.backward() #compute gradients
      #quick check before the update step if our sum is within our constraints
      if sum(torch.sum(torch.isnan(p.grad)) for p in model.parameters()) == 0:
        opt.step() #update


    total_err += (yp.max(dim = 1)[1] != y).sum().item() #compute error
    total_loss += loss.item() * X.shape[0]
    if monitor is not None:
      #total_monitor += monitor(model)
      #convert the monitor output into a tensor(rather than an int) if it isn't already
      #monitor_output = monitor(model)
      total_monitor += monitor(model) * X.shape[0] if isinstance(monitor(model), torch.Tensor) else float(monitor(model)) * X.shape[0]
  return total_err / len(loader.dataset), total_loss / len(loader.dataset), total_monitor / len(loader.dataset)


In [ ]:
#now train the model for 10 epochs:
for i in range(10):
  if i == 5:
    opt.param_groups[0]["lr"] = 1e-2


  train_err, train_loss, train_fpiter = epoch(train_loader, model, opt, lambda x : x[1].iterations)
  test_err, test_loss, test_fpiter = epoch(test_loader, model, monitor = lambda x : x[1].iterations)
  print(f"Train Error: {train_err: .4f}, Loss: {train_loss: .4f}, FP Iterations: {train_fpiter: .2f}" |
  + f"Test Error: {test_err: .4f}, Loss: {test_loss: .4f}, FP Iterations: {test_fpiter: .2f}")

  0%|          | 0/77 [00:00<?, ?it/s]